# Estado de Dados Brasil: Dashboards Executivos
**Tech Challenge - Fase 3 | Análise e Visualização Estratégica**

---

### 🎯 1. Objetivo
Este notebook consome diretamente as **7 SPECS Analíticas** pré-processadas em `dados/bases_analiticas/*.parquet` via **DuckDB** e **Plotly**. 

> **⚡ Performance Ultrarrápida**: Como as SPECS já foram transformadas e otimizadas pelo pipeline PySpark em `01_engenharia_specs_pyspark.ipynb`, este notebook não inicializa a JVM nem recria as bases, executando todas as consultas SQL e gerando os dashboards em **menos de 1 segundo**!

---

### 📊 Cobertura das 7 Perguntas Estratégicas:
1. **Estrutura do Mercado & Equipes** (`spec_respondentes` + `spec_estrutura_times_empresa`)
2. **Valorização Salarial & Prêmio por Stack** (`spec_respondentes` + `spec_adocao_tecnologias`)
3. **Diversidade de Gênero, Funil & Pay Gap** (`spec_diversidade_carreira`)
4. **Panorama Tecnológico Líder** (`spec_adocao_tecnologias`)
5. **Maturidade e Adoção de IA por Setor** (`spec_adocao_ia`)
6. **Dinâmica de Trabalho & Retenção de Talentos** (`spec_dinamica_trabalho_satisfacao` + `spec_segmentacao_negocio`)
7. **Barreiras de IA & Plano de Ação para Diretoria** (`spec_adocao_ia`)


In [ ]:
from pathlib import Path
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sqlglot
import pandas as pd

# Configuração de caminhos das SPECS
BASE_PROJETO = Path.cwd()
PASTA_SPECS = BASE_PROJETO / 'dados' / 'bases_analiticas'

if not PASTA_SPECS.exists():
    PASTA_SPECS = BASE_PROJETO / 'projeto' / 'fase_3_data_analytics' / 'dados' / 'bases_analiticas'

if not PASTA_SPECS.exists():
    PASTA_SPECS = Path('D:/d/pos/Fase 3/projeto/fase_3_data_analytics/dados/bases_analiticas')

assert PASTA_SPECS.exists(), f"Pasta de SPECS não encontrada: {PASTA_SPECS}"

# Inicializa conexão DuckDB em memória
con = duckdb.connect()

# Registra automaticamente todas as 7 SPECS como Views SQL
for f in PASTA_SPECS.glob('*.parquet'):
    nome = f.stem
    con.execute(f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_parquet('{f.as_posix()}')")

def executar_sql(query):
    query_validada = sqlglot.parse_one(query, read="duckdb")
    return con.sql(query_validada.sql(dialect="duckdb")).df()

print("[OK] Motor SQL DuckDB conectado instantaneamente a todas as 7 SPECS:")
for f in sorted(PASTA_SPECS.glob('*.parquet')):
    qtd = con.sql(f"SELECT COUNT(*) FROM {f.stem}").fetchone()[0]
    print(f"  - {f.name:<45} | Linhas: {qtd:,}")


## 🎯 Pergunta 1: Como está estruturado o mercado brasileiro de Dados?
* **Distribuição de Profissionais**: Liderado por **Análise de Dados & BI (25.8%)**, **Engenharia & Arquitetura de Dados (16.3%)** e **Ciência de Dados (13.2%)**.
* **Composição das Equipes de Dados nas Empresas**: **81.3%** das organizações contam com Analistas de Dados, **69.1%** possuem Engenheiros de Dados e **63.5%** Cientistas de Dados. Papéis mais especializados como *Machine Learning Engineer (25.9%)* e *Analytics Engineer (35.0%)* crescem aceleradamente em empresas maduras.


In [ ]:
df_cargos = executar_sql("""
    SELECT macro_cargo,
           COUNT(*) AS total_profissionais,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM spec_respondentes), 1) AS pct_mercado
    FROM spec_respondentes
    WHERE macro_cargo NOT IN ('Não informado / Em transição', 'Outros')
    GROUP BY 1 ORDER BY total_profissionais ASC
""")

df_equipes = executar_sql("""
    SELECT papel_na_empresa,
           COUNT(DISTINCT respondente_key) AS total_empresas,
           ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_estrutura_times_empresa), 1) AS penetracao_pct
    FROM spec_estrutura_times_empresa
    GROUP BY 1 ORDER BY total_empresas ASC
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Distribuição de Profissionais por Macro-Cargo</b>', '<b>Presença de Papéis de Dados nas Empresas Brasileiras</b>'),
    horizontal_spacing=0.14
)

fig.add_trace(go.Bar(
    x=df_cargos['total_profissionais'], y=df_cargos['macro_cargo'],
    orientation='h', marker_color='#1f77b4',
    text=df_cargos.apply(lambda r: f"{r['total_profissionais']:,} ({r['pct_mercado']}%)", axis=1),
    textposition='outside', name='Profissionais'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_equipes['penetracao_pct'], y=df_equipes['papel_na_empresa'],
    orientation='h', marker_color='#2ca02c',
    text=df_equipes.apply(lambda r: f"{r['penetracao_pct']}% (n={r['total_empresas']:,})", axis=1),
    textposition='outside', name='Penetração em Empresas'
), row=1, col=2)

fig.update_layout(
    template='plotly_white', height=480, showlegend=False,
    title={'text': '<b>Estrutura do Mercado e Organização dos Times de Dados</b>', 'font': {'size': 18}}
)
fig.update_xaxes(title='Total de Profissionais', row=1, col=1)
fig.update_xaxes(range=[0, 100], title='% das Organizações com o Papel', row=1, col=2)
fig.show()


## 🎯 Pergunta 2: Quais perfis profissionais são mais valorizados pelo mercado?
* **Remuneração por Cargo**: **Machine Learning & IA** recebe a maior remuneração média (R$ 16.186), seguida por **Engenharia de Dados** (R$ 12.603) e **Ciência de Dados** (R$ 11.560).
* **Prêmio Salarial por Stack**: Profissionais que dominam tecnologias de ponta em Lakehouse e Processamento Distribuído (**Redis R$ 16.068**, **Scala R$ 15.833**, **Elasticsearch R$ 14.805**, **Snowflake R$ 14.724**, **Databricks R$ 12.980**) têm remuneração substancialmente superior à média geral.


In [ ]:
df_nivel_cargo = executar_sql("""
    SELECT macro_cargo, nivel,
           ROUND(AVG(salario_estimado_num), 0) AS salario_medio
    FROM spec_respondentes
    WHERE salario_estimado_num IS NOT NULL 
      AND nivel IN ('Júnior', 'Pleno', 'Sênior', 'Especialista/Staff+')
      AND macro_cargo IN ('Engenharia & Arquitetura de Dados', 'Ciência de Dados', 'Análise de Dados & BI', 'Machine Learning & IA')
    GROUP BY 1, 2
    ORDER BY CASE nivel WHEN 'Júnior' THEN 1 WHEN 'Pleno' THEN 2 WHEN 'Sênior' THEN 3 WHEN 'Especialista/Staff+' THEN 4 END
""")

df_stack_premio = executar_sql("""
    SELECT tecnologia, familia,
           ROUND(AVG(salario_estimado_num), 0) AS media_salarial,
           COUNT(DISTINCT respondente_key) AS total_usuarios
    FROM spec_adocao_tecnologias
    WHERE salario_estimado_num IS NOT NULL
    GROUP BY 1, 2
    HAVING COUNT(DISTINCT respondente_key) >= 200
    ORDER BY media_salarial ASC LIMIT 8
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Remuneração Média (R$) por Senioridade</b>', '<b>Top Tecnologias com Maior Prêmio Salarial Médio</b>'),
    horizontal_spacing=0.14
)

cores = {'Engenharia & Arquitetura de Dados': '#2ca02c', 'Ciência de Dados': '#9467bd', 'Machine Learning & IA': '#ff7f0e', 'Análise de Dados & BI': '#17becf'}
for cargo in df_nivel_cargo['macro_cargo'].unique():
    sub = df_nivel_cargo[df_nivel_cargo['macro_cargo'] == cargo]
    fig.add_trace(go.Bar(
        x=sub['nivel'], y=sub['salario_medio'],
        name=cargo, marker_color=cores.get(cargo, '#333333'),
        text=sub['salario_medio'].apply(lambda v: f"R$ {v:,.0f}"), textposition='auto'
    ), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_stack_premio['media_salarial'], y=df_stack_premio['tecnologia'],
    orientation='h', marker_color='#f08c46',
    text=df_stack_premio.apply(lambda r: f"R$ {r['media_salarial']:,.0f} (n={r['total_usuarios']})", axis=1),
    textposition='inside', name='Salário Stack'
), row=1, col=2)

fig.update_layout(
    template='plotly_white', height=460, barmode='group',
    title={'text': '<b>Valorização Financeira por Especialidade Técnica e Stack</b>', 'font': {'size': 18}},
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5)
)
fig.show()


## 🎯 Pergunta 3: Qual é o cenário de diversidade de gênero nas carreiras de dados?
* **Sub-representação Geral**: Mulheres compõem apenas **23.6%** do mercado de dados.
* **Efeito Funil ("Teto de Vidro")**: A participação feminina sofre declínio contínuo conforme a senioridade avança: **27.2% no nível Júnior**, **24.8% no Pleno**, **20.5% no Sênior** e apenas **16.2% em Especialista/Staff+**.
* **Disparidade Salarial (Pay Gap)**: A diferença salarial entre homens e mulheres atinge **15.6% no nível Sênior** e **14.9% no nível Especialista/Staff+**.


In [ ]:
df_funil = executar_sql("""
    SELECT nivel, 
           COUNT(*) AS total,
           ROUND(100.0 * SUM(CASE WHEN genero = 'Feminino' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_mulheres,
           ROUND(AVG(CASE WHEN genero = 'Masculino' THEN salario_estimado_num END), 0) AS sal_homens,
           ROUND(AVG(CASE WHEN genero = 'Feminino' THEN salario_estimado_num END), 0) AS sal_mulheres,
           ROUND(100.0 * (AVG(CASE WHEN genero = 'Masculino' THEN salario_estimado_num END) - AVG(CASE WHEN genero = 'Feminino' THEN salario_estimado_num END)) / AVG(CASE WHEN genero = 'Masculino' THEN salario_estimado_num END), 1) AS pay_gap_pct
    FROM spec_respondentes 
    WHERE nivel IN ('Júnior', 'Pleno', 'Sênior', 'Especialista/Staff+')
      AND genero IN ('Masculino', 'Feminino')
    GROUP BY nivel 
    ORDER BY CASE nivel WHEN 'Júnior' THEN 1 WHEN 'Pleno' THEN 2 WHEN 'Sênior' THEN 3 WHEN 'Especialista/Staff+' THEN 4 END
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Funil de Representatividade Feminina por Senioridade</b>', '<b>Comparativo Salarial Médio (Homens vs Mulheres)</b>'),
    horizontal_spacing=0.12
)

fig.add_trace(go.Bar(
    x=df_funil['nivel'], y=df_funil['pct_mulheres'],
    marker_color='#e377c2', name='% Mulheres',
    text=df_funil['pct_mulheres'].apply(lambda v: f"{v}%"), textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_funil['nivel'], y=df_funil['sal_homens'],
    name='Homens (R$)', marker_color='#1f77b4',
    text=df_funil['sal_homens'].apply(lambda v: f"R$ {v:,.0f}"), textposition='auto'
), row=1, col=2)

fig.add_trace(go.Bar(
    x=df_funil['nivel'], y=df_funil['sal_mulheres'],
    name='Mulheres (R$)', marker_color='#e377c2',
    text=df_funil['sal_mulheres'].apply(lambda v: f"R$ {v:,.0f}"), textposition='auto'
), row=1, col=2)

fig.update_layout(
    template='plotly_white', height=450,
    title={'text': '<b>Cenário de Diversidade de Gênero e Equidade Salarial</b>', 'font': {'size': 18}},
    barmode='group', showlegend=True, legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5)
)
fig.update_yaxes(range=[0, 35], title='% Representação Feminina', row=1, col=1)
fig.update_yaxes(title='Salário Médio (R$)', row=1, col=2)
fig.show()


## 🎯 Pergunta 4: Quais tecnologias apresentam maior adoção entre os profissionais?
* **Linguagens**: **SQL (57.6%)** e **Python (54.9%)** constituem o core do mercado brasileiro de dados.
* **Cloud Providers**: **AWS (26.4%)**, **Azure (24.0%)** e **Google Cloud (20.3%)** compõem um ecossistema corporativo multi-cloud equilibrado.
* **Bancos & Plataformas**: **PostgreSQL (21.2%)** e **SQL Server (21.2%)** lideram o relacional; **Databricks (18.7%)**, **BigQuery (15.5%)** e **Snowflake** dominam o processamento analítico moderno.


In [ ]:
df_tech = executar_sql("""
    WITH ranked AS (
        SELECT familia, tecnologia,
               COUNT(DISTINCT respondente_key) AS adotantes,
               ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_respondentes), 1) AS pct_mercado,
               ROW_NUMBER() OVER(PARTITION BY familia ORDER BY COUNT(DISTINCT respondente_key) DESC) AS rnk
        FROM spec_adocao_tecnologias
        GROUP BY 1, 2
    )
    SELECT familia, tecnologia, adotantes, pct_mercado
    FROM ranked WHERE rnk <= 5
    ORDER BY familia, pct_mercado DESC
""")

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('<b>Linguagens de Programação</b>', '<b>Cloud Providers</b>',
                    '<b>Bancos de Dados & Data Platforms</b>', '<b>Ferramentas de ETL & Orquestração</b>'),
    vertical_spacing=0.15, horizontal_spacing=0.12
)

def add_family_bar(familia, row, col, color):
    sub = df_tech[df_tech['familia'] == familia].sort_values('pct_mercado', ascending=True)
    fig.add_trace(go.Bar(
        x=sub['pct_mercado'], y=sub['tecnologia'],
        orientation='h', marker_color=color,
        text=sub['pct_mercado'].apply(lambda v: f"{v}%"), textposition='outside'
    ), row=row, col=col)

add_family_bar('linguagem', 1, 1, '#1f77b4')
add_family_bar('cloud', 1, 2, '#ff7f0e')
add_family_bar('banco_ou_plataforma', 2, 1, '#2ca02c')
add_family_bar('etl', 2, 2, '#d62728')

fig.update_layout(
    template='plotly_white', height=650, showlegend=False,
    title={'text': '<b>Stack Tecnológico Líder no Mercado Brasileiro de Dados</b>', 'font': {'size': 18}}
)
fig.update_xaxes(range=[0, 70])
fig.show()


## 🎯 Pergunta 5: Qual é o índice de adoção de Inteligência Artificial e seu impacto?
* **Adoção Geral**: **64.3% dos profissionais** já adotam soluções de IA ativamente.
* **Maturidade Setorial**: **Consultorias (34.1%)**, **Bancos/Fintechs (33.4%)** e **Tecnologia (33.2%)** lideram a aplicação corporativa em processos e produtos para clientes.
* **Tipo de Investimento**: 35.6% utilizam ferramentas gratuitas e 13.8% pagam assinaturas do próprio bolso, evidenciando que os colaboradores estão puxando a inovação antes mesmo das políticas corporativas formais.


In [ ]:
df_ia_cat = executar_sql("""
    SELECT categoria_ia, indicador_ia_label,
           COUNT(DISTINCT respondente_key) AS adotantes,
           ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_respondentes), 1) AS pct_mercado
    FROM spec_adocao_ia
    WHERE tipo_registro = 'Uso Ativo'
    GROUP BY 1, 2
    ORDER BY adotantes ASC LIMIT 8
""")

df_ia_setor = executar_sql("""
    SELECT r.setor,
           COUNT(DISTINCT r.respondente_key) AS total,
           ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.indicador_ia_id IN ('ia_produtos_internos', 'ia_produtos_externos', 'ia_principal_frente_negocio') THEN a.respondente_key END) / COUNT(DISTINCT r.respondente_key), 1) AS pct_ia_corporativa
    FROM spec_respondentes r
    LEFT JOIN spec_adocao_ia a ON r.respondente_key = a.respondente_key AND a.tipo_registro = 'Uso Ativo'
    WHERE r.setor IS NOT NULL
    GROUP BY 1 HAVING COUNT(DISTINCT r.respondente_key) >= 400
    ORDER BY pct_ia_corporativa ASC
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Modalidades de Uso de IA no Dia a Dia</b>', '<b>Taxa de IA Integrada a Produtos/Processos por Setor</b>'),
    horizontal_spacing=0.14
)

fig.add_trace(go.Bar(
    x=df_ia_cat['pct_mercado'], y=df_ia_cat['indicador_ia_label'],
    orientation='h', marker_color='#0b7285',
    text=df_ia_cat.apply(lambda r: f"{r['pct_mercado']}%", axis=1), textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_ia_setor['pct_ia_corporativa'], y=df_ia_setor['setor'],
    orientation='h', marker_color='#f08c46',
    text=df_ia_setor.apply(lambda r: f"{r['pct_ia_corporativa']}%", axis=1), textposition='outside'
), row=1, col=2)

fig.update_layout(
    template='plotly_white', height=460, showlegend=False,
    title={'text': '<b>Índice de Adoção e Maturidade Organizacional de IA</b>', 'font': {'size': 18}}
)
fig.update_xaxes(range=[0, 45], title='% de Penetração no Mercado', row=1, col=1)
fig.update_xaxes(range=[0, 45], title='% com IA em Processos/Produtos', row=1, col=2)
fig.show()


## 🎯 Pergunta 6: Existem diferenças relevantes entre regiões, senioridades ou modelos de trabalho?
* **Descompasso Trabalho Atual vs Ideal**: **48.3% dos profissionais** em modelos presenciais ou híbridos fixos desejam maior flexibilidade (Híbrido Flexível ou Remoto). Somente **1.2%** dos profissionais consideram o regime 100% presencial como ideal.
* **Adoção Tecnológica e Salário**: Profissionais em regime remoto apresentam média salarial 35% superior aos presenciais e maior taxa de adesão a ferramentas modernas de nuvem e IA.


In [ ]:
df_ia_rotina = executar_sql("""
    SELECT COALESCE(modelo_trabalho_resumido, 'Não informado') AS modelo,
           COUNT(DISTINCT r.respondente_key) AS total,
           COUNT(DISTINCT a.respondente_key) AS adotantes,
           ROUND(100.0 * COUNT(DISTINCT a.respondente_key) / COUNT(DISTINCT r.respondente_key), 1) AS pct_adocao
    FROM spec_respondentes r
    LEFT JOIN (SELECT DISTINCT respondente_key FROM spec_adocao_ia WHERE tipo_registro = 'Uso Ativo') a 
      ON r.respondente_key = a.respondente_key
    WHERE modelo_trabalho_resumido != 'Não informado'
    GROUP BY 1 ORDER BY pct_adocao ASC
""")

df_alinhamento = executar_sql("""
    SELECT status_alinhamento,
           COUNT(*) AS total,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM spec_dinamica_trabalho_satisfacao WHERE status_alinhamento != 'Não informado'), 1) AS pct
    FROM spec_dinamica_trabalho_satisfacao
    WHERE status_alinhamento != 'Não informado'
    GROUP BY 1 ORDER BY total ASC
""")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Taxa de Adoção de IA por Regime de Trabalho</b>', '<b>Alinhamento entre Modelo Atual vs Ideal (Risco de Retenção)</b>'),
    horizontal_spacing=0.14
)

fig.add_trace(go.Bar(
    x=df_ia_rotina['pct_adocao'], y=df_ia_rotina['modelo'],
    orientation='h', marker_color='#0b7285',
    text=df_ia_rotina.apply(lambda r: f"{r['pct_adocao']}% (n={r['total']:,})", axis=1), textposition='inside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_alinhamento['pct'], y=df_alinhamento['status_alinhamento'],
    orientation='h', marker_color='#845ef7',
    text=df_alinhamento.apply(lambda r: f"{r['pct']}% (n={r['total']:,})", axis=1), textposition='inside'
), row=1, col=2)

fig.update_layout(
    template='plotly_white', height=450, showlegend=False,
    title={'text': '<b>Dinâmica de Trabalho, Adoção Tecnológica e Retenção de Talentos</b>', 'font': {'size': 18}}
)
fig.update_xaxes(range=[0, 100], title='% Adoção de IA', row=1, col=1)
fig.update_xaxes(range=[0, 100], title='% dos Profissionais', row=1, col=2)
fig.show()


## 🎯 Pergunta 7: Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e IA?
* **Principais Barreiras**: **Falta de Expertise Técnica (36.8%)**, **Falta de Compreensão dos Casos de Uso (36.0%)** e **Dados Corporativos Despreparados / Falta de Governança (33.1%)**.
* **Recomendações Estratégicas para C-Level**:
  1. **Capacitação & Governança Primeiro**: Antes de investir em ferramentas caras de IA, estruturar o catálogo, linhagem e qualidade de dados.
  2. **Atração de Talentos**: Oferecer políticas claras de flexibilidade (Remoto/Híbrido) para reter os melhores profissionais e combater o turnover de 48%.
  3. **Equidade de Gênero**: Criar programas ativos de mentoria e revisão salarial para quebrar o teto de vidro nos níveis seniores.


In [ ]:
df_barreiras = executar_sql("""
    SELECT indicador_ia_label AS barreira,
           COUNT(DISTINCT respondente_key) AS ocorrencias,
           ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_adocao_ia WHERE tipo_registro = 'Barreira'), 1) AS pct_barreiras
    FROM spec_adocao_ia
    WHERE tipo_registro = 'Barreira'
    GROUP BY 1 ORDER BY ocorrencias ASC
""")

resumo_diretoria = pd.DataFrame([
    {"Pilar Estratégico": "Pessoas & Diversidade", 
     "Diagnóstico dos Dados": "Mulheres representam 23.6% do mercado com teto de vidro nos níveis seniores (16.2% Staff+) e pay gap de 15.6%.", 
     "Plano de Ação Executivo": "Auditar remuneração por nível, estabelecer metas afirmativas em contratação e criar programas de mentoria para liderança técnica feminina."},
    
    {"Pilar Estratégico": "Stack Tecnológico", 
     "Diagnóstico dos Dados": "SQL (57.6%) e Python (54.9%) lideram o mercado, acompanhados de ecossistemas em nuvem (AWS/Azure/GCP).", 
     "Plano de Ação Executivo": "Consolidar arquitetura Lakehouse (Databricks/Snowflake/Cloud) e padronizar pipelines de dados com Python/SQL para acelerar o time to market analítico."},
    
    {"Pilar Estratégico": "Investimento em IA", 
     "Diagnóstico dos Dados": "64.3% dos profissionais utilizam IA, mas esbarram em falta de expertise (36.8%) e dados corporativos despreparados (33.1%).", 
     "Plano de Ação Executivo": "Priorizar investimentos em Governança, Qualidade de Dados e letramento corporativo em IA antes de adquirir ferramentas pontuais de alto custo."}
])

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.55, 0.45],
    specs=[[{"type": "xy"}], [{"type": "table"}]],
    vertical_spacing=0.15,
    subplot_titles=('<b>Principais Barreiras para Escala de IA nas Empresas</b>', '<b>Plano de Ação Executivo para a Diretoria</b>')
)

fig.add_trace(go.Bar(
    x=df_barreiras['pct_barreiras'], y=df_barreiras['barreira'],
    orientation='h', marker_color='#c92a2a',
    text=df_barreiras.apply(lambda r: f"{r['pct_barreiras']}% ({r['ocorrencias']:,} menções)", axis=1),
    textposition='inside'
), row=1, col=1)

fig.add_trace(go.Table(
    columnorder=[1, 2, 3],
    columnwidth=[20, 38, 42],
    header=dict(
        values=[f"<b>{c}</b>" for c in resumo_diretoria.columns],
        fill_color='#1f77b4', font=dict(color='white', size=13), align='left', height=30
    ),
    cells=dict(
        values=[resumo_diretoria[k] for k in resumo_diretoria.columns],
        fill_color='#f8f9fa', font=dict(size=12), align='left', height=45
    )
), row=2, col=1)

fig.update_layout(
    template='plotly_white', height=750, showlegend=False,
    title={'text': '<b>Diagnóstico de Risco e Roadmap Estratégico de IA</b>', 'font': {'size': 18}}
)
fig.update_xaxes(range=[0, 45], title='% das Empresas com Barreiras Relatadas', row=1, col=1)
fig.show()
